# Getting started with pounce

`pounce` is a Rust port of Ipopt for nonlinear optimization. The Python interface gives you two entry points:

* **`pounce.Problem`** — the cyipopt-shaped object-oriented API. You hand the solver a user object with `objective`/`gradient`/`constraints`/`jacobian`/`hessian` methods.
* **`pounce.minimize`** — the scipy-style functional facade. Pass `fun`, `jac`, `hess`, `bounds`, `constraints` directly.

Both build on the same Rust solver. This notebook walks both paths plus the warm-start convention.

We'll use **HS071** — the canonical Ipopt test problem — throughout:

$$\min_{x \in \mathbb{R}^4} \; x_1 x_4 (x_1 + x_2 + x_3) + x_3$$

$$\text{s.t.} \quad x_1 x_2 x_3 x_4 \geq 25, \quad x_1^2 + x_2^2 + x_3^2 + x_4^2 = 40, \quad 1 \leq x_i \leq 5.$$

Optimum: $x^\star \approx (1.000, 4.743, 3.821, 1.379)$, $f^\star \approx 17.014$.

In [1]:
import numpy as np

import pounce
print(f"pounce {pounce.__version__}")

pounce 0.4.0


## 1. The `Problem` API (cyipopt-style)

Build a class whose methods match the cyipopt calling convention. The solver detects whether you supplied an exact Hessian via `hasattr(obj, 'hessian')` and falls back to L-BFGS otherwise.

Sparse Jacobians and Hessians use the `(rows, cols)` structure / `values` value split, also cyipopt-style:

In [2]:
class HS071:
    def objective(self, x):
        return x[0] * x[3] * (x[0] + x[1] + x[2]) + x[2]

    def gradient(self, x):
        return np.array([
            x[0] * x[3] + x[3] * (x[0] + x[1] + x[2]),
            x[0] * x[3],
            x[0] * x[3] + 1.0,
            x[0] * (x[0] + x[1] + x[2]),
        ])

    def constraints(self, x):
        return np.array([np.prod(x), np.dot(x, x)])

    def jacobianstructure(self):
        # Dense 2-by-4: both constraints depend on all variables.
        return (np.repeat([0, 1], 4), np.tile([0, 1, 2, 3], 2))

    def jacobian(self, x):
        return np.array([
            x[1] * x[2] * x[3], x[0] * x[2] * x[3],
            x[0] * x[1] * x[3], x[0] * x[1] * x[2],
            2 * x[0], 2 * x[1], 2 * x[2], 2 * x[3],
        ])

In [3]:
prob = pounce.Problem(
    n=4, m=2, problem_obj=HS071(),
    lb=[1.0] * 4, ub=[5.0] * 4,
    cl=[25.0, 40.0], cu=[2e19, 40.0],
)
prob.add_option("tol", 1e-8)
prob.add_option("print_level", 0)

x, info = prob.solve(x0=np.array([1.0, 5.0, 5.0, 1.0]))
print(f"status: {info['status_msg']}")
print(f"x*    : {x}")
print(f"f*    : {info['obj_val']:.6f}")
print(f"iters : {info['iter_count']}")

status: Solve_Succeeded
x*    : [0.99999999 4.74299964 3.82114998 1.37940829]
f*    : 17.014017
iters : 11


### What's in `info`?

Everything you'd expect from a KKT solve:

* `status`, `status_msg` — exit code / human-readable reason
* `obj_val`, `g` — final objective and constraint residuals
* `mult_g` — equality / inequality constraint multipliers (the Lagrange $\lambda$)
* `mult_x_L`, `mult_x_U` — bound multipliers ($z^L$, $z^U$)
* `iter_count` — iteration count

In [4]:
for k, v in info.items():
    print(f"{k:>12} : {v}")

      status : 0
  status_msg : Solve_Succeeded
     obj_val : 17.014017145215984
           g : [24.99999975 40.        ]
      mult_g : [-0.55229366  0.16146856]
    mult_x_L : [1.08787123e+00 6.69490727e-10 8.88256054e-10 6.60476737e-09]
    mult_x_U : [6.26475881e-10 9.75058202e-09 2.12571864e-09 6.92125411e-10]
  iter_count : 11
          mu : 2.5059035596800634e-09
final_kkt_error : 2.5059035804827257e-09
final_dual_inf : 2.8998663518644377e-11
final_constr_viol : 3.552713678800501e-15
 final_compl : 2.5059035804827257e-09
 pinned_vars : [ True False False False]
active_constraints : [ True  True]
  active_tol : 1e-06
 working_set : None


## 2. The `minimize` facade (scipy-style)

`pounce.minimize` mirrors `scipy.optimize.minimize` closely enough to be a drop-in replacement for a local NLP solve. It returns a genuine **`scipy.optimize.OptimizeResult`** (with a pounce back-compat shim), so `res.x`, `res.fun`, `res.success`, `res.nit`, and the `res.nfev` / `res.njev` / `res.nhev` evaluation counters all work; pounce-specific extras live under `res.info`.

The scipy-compatible surface includes:

* **`args=(...)`** — extra positional arguments forwarded to `fun` / `jac`.
* **`jac=True`** — `fun` returns `(value, gradient)` in one call (cached, so the gradient is not recomputed).
* **`callback`** — invoked each iteration (both scipy signatures: `callback(xk)` and `callback(intermediate_result)`).
* **scipy `Bounds` and `LinearConstraint`** objects, alongside `(lo, hi)` pairs and constraint dicts.
* **scipy option spellings** as synonyms — `maxiter`, `gtol` / `ftol` / `xtol`, `disp`, `maxcor` map onto pounce / Ipopt names.
* **`method=pounce.minimize`** — usable directly inside `scipy.optimize.minimize`.

We start with the basic scipy-dict constraint form, then walk each feature in turn.

In [5]:
def f(x):
    return float(x @ x)

def grad(x):
    return 2.0 * x

def c_fun(x):
    return np.array([x[0] + x[1] - 1.0])

def c_jac(x):
    return np.array([[1.0, 1.0]])

# Constraints can be scipy-style dicts: {"type": "eq"|"ineq", "fun", "jac"}.
# Options are plain keyword arguments (the legacy options={...} dict still works too).
res = pounce.minimize(
    f, x0=np.zeros(2), jac=grad,
    constraints=[{"type": "eq", "fun": c_fun, "jac": c_jac}],
    tol=1e-10, print_level=0,
)
print(f"success: {res.success}")
print(f"x*     : {res.x}")
print(f"f*     : {res.fun:.6f}")
print(f"iters  : {res.nit},  nfev: {res.nfev},  njev: {res.njev}")

success: True
x*     : [0.5 0.5]
f*     : 0.500000
iters  : 2,  nfev: 4,  njev: 4


### 2.1 scipy `Bounds` and `LinearConstraint`

Pass scipy's `Bounds` and `LinearConstraint` objects straight through — no need to translate them into `(lo, hi)` pairs or constraint dicts. A `LinearConstraint(A, lb, ub)` becomes the row constraints `lb <= A @ x <= ub` (one-sided bounds use `±np.inf`).

In [6]:
from scipy.optimize import Bounds, LinearConstraint

# min (x0 - 1)^2 + (x1 - 2.5)^2   s.t.   x0 - 2*x1 >= -2,   0 <= x <= 10
fun = lambda x: (x[0] - 1.0) ** 2 + (x[1] - 2.5) ** 2
jac = lambda x: np.array([2.0 * (x[0] - 1.0), 2.0 * (x[1] - 2.5)])

res = pounce.minimize(
    fun, x0=[2.0, 0.0], jac=jac,
    bounds=Bounds(0.0, 10.0),                                   # scipy Bounds object
    constraints=LinearConstraint([[1.0, -2.0]], lb=-2.0, ub=np.inf),
    tol=1e-9, print_level=0,
)
print(f"x* = {res.x}")
print(f"f* = {res.fun:.6f}")

x* = [1.4        1.70000001]
f* = 0.800000


### 2.2 `args`, `jac=True`, and `callback`

Three scipy conventions in one solve:

* **`args=(a, b)`** forwards extra positional arguments to `fun` (and `jac`).
* **`jac=True`** means `fun` returns `(value, gradient)` together — pounce caches the pair so the gradient is never recomputed for the same `x`.
* **`callback`** fires once per iteration. pounce accepts both scipy signatures; here we use `callback(xk)` to record the iterate trajectory.

In [7]:
def rosen_with_grad(x, a, b):
    # Returns (value, gradient) together because we pass jac=True.
    val = (a - x[0]) ** 2 + b * (x[1] - x[0] ** 2) ** 2
    g = np.array([
        -2 * (a - x[0]) - 4 * b * x[0] * (x[1] - x[0] ** 2),
        2 * b * (x[1] - x[0] ** 2),
    ])
    return val, g

trajectory = []
def cb(xk):
    trajectory.append(np.copy(xk))

res = pounce.minimize(
    rosen_with_grad, x0=[-1.2, 1.0],
    args=(1.0, 100.0),   # a=1, b=100  ->  classic Rosenbrock
    jac=True,            # fun returns (f, grad)
    callback=cb,
    tol=1e-10, print_level=0,
)
print(f"x* = {res.x},  f* = {res.fun:.3e}")
print(f"nfev = {res.nfev}, njev = {res.njev}, callback saw {len(trajectory)} iterates")

x* = [1. 1.],  f* = 3.385e-28
nfev = 80, njev = 42, callback saw 41 iterates


### 2.3 scipy option names and evaluation counters

You can spell options the scipy way and pounce maps them onto its own (Ipopt) names: `maxiter` → `max_iter`, `gtol` / `ftol` / `xtol` → `tol`, `disp` → `print_level`, `maxcor` → `limited_memory_max_history`. The result carries scipy's `nfev` / `njev` / `nhev` evaluation counters.

In [8]:
res = pounce.minimize(
    lambda x: x @ x, x0=np.ones(5), jac=lambda x: 2.0 * x,
    maxiter=100,   # scipy spelling -> max_iter
    gtol=1e-9,     # scipy spelling -> tol
    disp=False,    # scipy spelling -> print_level
)
print(f"x*   = {res.x}")
print(f"nfev = {res.nfev}, njev = {res.njev}, nhev = {res.nhev}, nit = {res.nit}")

x*   = [0. 0. 0. 0. 0.]
nfev = 8, njev = 3, nhev = 0, nit = 1


### 2.4 Drop-in for `scipy.optimize.minimize(method=...)`

`pounce.minimize` satisfies scipy's *custom minimizer* contract, so you can hand it to `scipy.optimize.minimize` as the `method=` argument. Existing scipy call sites keep working unchanged — they are just solved by pounce's filter interior-point method.

In [9]:
from scipy.optimize import minimize as scipy_minimize

res = scipy_minimize(
    lambda x: (x[0] - 3) ** 2 + (x[1] + 1) ** 2,
    x0=[0.0, 0.0],
    method=pounce.minimize,                                  # <- pounce as the backend
    jac=lambda x: np.array([2 * (x[0] - 3), 2 * (x[1] + 1)]),
)
print(f"x* = {res.x},  f* = {res.fun:.3e}")

x* = [ 3. -1.],  f* = 0.000e+00


### 2.5 Automatic convex routing (`solver_selection`)

By default `minimize` always uses the general NLP solver — no structure probing, so an expensive `fun` pays nothing. Opt in with `solver_selection="auto"` and pounce probes the callables: a problem that is provably a **linear program** or **convex quadratic program** is dispatched to the dedicated convex interior-point solver, reaching a **global** optimum in far fewer iterations. Detection is validated against the true callables, so a nonlinear problem is never silently mis-routed.

When a solve is routed to the convex solver the python `fun`/`jac` callbacks no longer fire (the solver consumes the extracted matrix form), so `res.nfev` is `0` and `res.info["solver"]` names the engine that ran.

In [10]:
res = pounce.minimize(
    lambda x: x @ x, x0=[1.0, 1.0], jac=lambda x: 2.0 * x,
    bounds=[(-1, 1), (-1, 1)],
    solver_selection="auto",     # opt into structure detection
)
print(f"routed to : {res.info.get('solver')}")   # 'qp-ipm'
print(f"x* = {res.x}")
print(f"nfev = {res.nfev}  (0 -> the convex solver consumed the extracted QP)")

routed to : qp-ipm
x* = [-3.94806408e-12 -3.94806408e-12]
nfev = 0  (0 -> the convex solver consumed the extracted QP)


## 3. L-BFGS path (no Hessian supplied)

Omit `hessian`/`hessianstructure` from your problem object and the solver runs with a limited-memory quasi-Newton approximation. The same `Problem` interface — pounce auto-switches based on what your object provides.

In [11]:
class HS071_no_hess(HS071):
    pass  # No hessian / hessianstructure → L-BFGS path.

prob = pounce.Problem(
    n=4, m=2, problem_obj=HS071_no_hess(),
    lb=[1.0] * 4, ub=[5.0] * 4,
    cl=[25.0, 40.0], cu=[2e19, 40.0],
)
print(f"has_hessian: {prob.has_hessian}")

prob.add_option("tol", 1e-8)
prob.add_option("print_level", 0)
x_lbfgs, info_lbfgs = prob.solve(x0=np.array([1.0, 5.0, 5.0, 1.0]))
print(f"status: {info_lbfgs['status_msg']}, f* = {info_lbfgs['obj_val']:.6f}, iters = {info_lbfgs['iter_count']}")

has_hessian: False
status: Solve_Succeeded, f* = 17.014017, iters = 11


## 4. Warm-starting

Re-solving from a point near the optimum *can* cut iterations sharply — but `warm_start_init_point=yes` **alone does not help**. Forwarding the dual seeds is necessary but not sufficient, because two other defaults cancel the warm start:

- **`mu_init`** (default `0.1`) keeps the barrier parameter large, so the solver still walks μ down its full schedule even when started at $x^\star$.
- **`warm_start_bound_push` / `warm_start_bound_frac`** (default `1e-2`) push the initial point *off* its bounds — HS071's $x_1$ sits exactly on its lower bound (`1.0`), so the warm point is relaxed away and discarded.

This is upstream Ipopt's warm-start convention, which pounce follows exactly. To actually get the speedup, forward the dual seeds (`lagrange`, `zl`, `zu`) **and** set a small `mu_init` together with tight `warm_start_*_bound_push` / `_frac` values, so the iterate stays on its bounds and the barrier starts near convergence:

In [12]:
def make(extra=None):
    p = pounce.Problem(
        n=4, m=2, problem_obj=HS071(),
        lb=[1.0] * 4, ub=[5.0] * 4,
        cl=[25.0, 40.0], cu=[2e19, 40.0],
    )
    p.add_option("tol", 1e-8)
    p.add_option("print_level", 0)
    for k, v in (extra or {}).items():
        p.add_option(k, v)
    return p

cold_x, cold_info = make().solve(x0=np.array([1.0, 5.0, 5.0, 1.0]))

# `warm_start_init_point=yes` ALONE does not cut iterations: the default
# mu_init (0.1) and warm_start_*_bound_push (1e-2) negate the warm point.
naive_x, naive_info = make({"warm_start_init_point": "yes"}).solve(
    x0=cold_x,
    lagrange=np.asarray(cold_info["mult_g"]),
    zl=np.asarray(cold_info["mult_x_L"]),
    zu=np.asarray(cold_info["mult_x_U"]),
)

# The real warm start: small mu_init + tight bound pushes, so the iterate
# stays on its bounds and the barrier starts near convergence.
warm_opts = {
    "warm_start_init_point": "yes",
    "mu_init": 1e-7,
    "warm_start_bound_push": 1e-9,
    "warm_start_bound_frac": 1e-9,
    "warm_start_slack_bound_push": 1e-9,
    "warm_start_slack_bound_frac": 1e-9,
    "warm_start_mult_bound_push": 1e-9,
}
warm_x, warm_info = make(warm_opts).solve(
    x0=cold_x,
    lagrange=np.asarray(cold_info["mult_g"]),
    zl=np.asarray(cold_info["mult_x_L"]),
    zu=np.asarray(cold_info["mult_x_U"]),
)

print(f"cold (from [1,5,5,1])            : {cold_info['iter_count']} iters")
print(f"warm_start_init_point=yes only   : {naive_info['iter_count']} iters  (no gain)")
print(f"warm + mu_init + bound_push      : {warm_info['iter_count']} iters")
print(f"|Δx| (warm vs cold solution)      = {np.max(np.abs(warm_x - cold_x)):.2e}")

cold (from [1,5,5,1])            : 11 iters
warm_start_init_point=yes only   : 11 iters  (no gain)
warm + mu_init + bound_push      : 5 iters
|Δx| (warm vs cold solution)      = 2.19e-09


## 5. Setting solver options

Any Ipopt option name works. Strings, ints, floats, and bools are routed to the right setter automatically.

Common ones:

| Option | Effect |
|---|---|
| `tol` | Convergence tolerance on the KKT error |
| `max_iter` | Iteration cap |
| `print_level` | 0 = silent, 5 = default, 12 = noisy |
| `hessian_approximation` | `"exact"` (default) or `"limited-memory"` |
| `linear_solver` | Backend (`"feral"` is the pure-Rust default) |
| `mu_strategy` | `"monotone"` or `"adaptive"` |

See the [pounce **Solver Options** reference](https://kitchingroup.cheme.cmu.edu/pounce/options.html) for the full list with pounce's defaults and semantics — it documents the μ-strategy, scaling, FBBT, and FERAL knobs, and links the upstream [Ipopt options catalogue](https://coin-or.github.io/Ipopt/OPTIONS.html) whose option names pounce reuses.

In [13]:
prob = pounce.Problem(
    n=4, m=2, problem_obj=HS071(),
    lb=[1.0] * 4, ub=[5.0] * 4,
    cl=[25.0, 40.0], cu=[2e19, 40.0],
)
prob.add_option("tol", 1e-12)
prob.add_option("max_iter", 200)
prob.add_option("mu_strategy", "adaptive")
prob.add_option("print_level", 0)

x, info = prob.solve(x0=np.array([1.0, 5.0, 5.0, 1.0]))
print(info["status_msg"], info["iter_count"], info["obj_val"])

Solved_To_Acceptable_Level 21 17.014017140224176
